# Qwen3 top-k teacher logits on TPU

Produces the distillation artefact described in `docs/tasks/tpu-topk-teacher.md`:
one record per **teacher** token — top-k ids, top-k values, and the token's byte
offset — so a 32k student can align by byte span later.

**Refuses to run without a TPU.** A CPU run is not a substitute and is not
recorded as a measurement.

Order of work: smoke (0.6B, 1M tokens, k=8) → real (1.7B, 22.7M tokens, k=32).


In [ ]:
import os, shutil, subprocess, sys

# A TPU VM has /dev/accel* or the XRT device; a CPU session has neither.
has_tpu = bool(os.environ.get('TPU_NAME')) or bool(shutil.os.listdir('/dev') and
    [d for d in os.listdir('/dev') if d.startswith(('accel', 'vfal'))])
if not has_tpu:
    raise SystemExit(
        'no TPU device in this session. Settings -> Accelerator -> TPU VM v3-8, '
        'then Restart Session. A CPU run is not an acceptable substitute.'
    )
print('TPU device present')
print(subprocess.run(['bash', '-lc', 'ls -l /dev/accel* 2>/dev/null; printenv | grep -i -E "TPU|XRT" | head'],
                     capture_output=True, text=True).stdout)


In [ ]:
!pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html 2>&1 | tail -3
import torch, torch_xla
import torch_xla.core.xla_model as xm
print('torch', torch.__version__, 'xla device:', xm.xla_device())


In [ ]:
# SMOKE=1 runs the 0.6B teacher on a small slice first. Do not skip it.
SMOKE = os.environ.get('SMOKE', '1') == '1'
TEACHER = 'Qwen/Qwen3-0.6B' if SMOKE else 'Qwen/Qwen3-1.7B'
K = 8 if SMOKE else 32
MAX_TOKENS = 1_000_000 if SMOKE else 22_700_000

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(TEACHER)
model = AutoModelForCausalLM.from_pretrained(TEACHER, torch_dtype=torch.bfloat16)
model = model.to(xm.xla_device()).eval()
print('teacher', TEACHER, 'vocab', len(tok), 'k', K, 'budget', MAX_TOKENS)


In [ ]:
# The corpus shards hold u16 ids from a lossless byte tokenizer, so the text is
# recoverable exactly. Assert that before trusting a single teacher token.
import glob, struct
from pathlib import Path

CORPUS = sorted(glob.glob('/kaggle/input/*/**/*.u16', recursive=True))
if not CORPUS:
    CORPUS = sorted(glob.glob('/kaggle/input/**/*.bin', recursive=True))
assert CORPUS, 'attach the corpus dataset; no shard found under /kaggle/input'
print('shards:', len(CORPUS), 'first:', CORPUS[0])

def read_u16(path):
    raw = Path(path).read_bytes()
    return list(struct.unpack('<%dH' % (len(raw) // 2), raw))

def byte_decode(ids):
    # byte tokenizer: id == byte value
    return bytes(i for i in ids).decode('utf-8', errors='strict')

ids = read_u16(CORPUS[0])[:200_000]
text = byte_decode(ids)
back = [b for b in text.encode('utf-8')]
assert back == ids[:len(back)], 'byte tokenizer round trip failed; stop'
print('round trip ok:', len(ids), 'ids ->', len(text), 'chars')


In [ ]:
import time, json

out_dir = Path('/kaggle/working/topk')
out_dir.mkdir(exist_ok=True)
enc = tok(text, return_offsets_mapping=True, add_special_tokens=False)
input_ids = torch.tensor([enc['input_ids']], dtype=torch.long)
offsets = enc['offset_mapping']

t_compile = time.perf_counter()
with torch.no_grad():
    out = model(input_ids.to(xm.xla_device()))
xm.mark_step()
compile_s = time.perf_counter() - t_compile

logits = out.logits[0].float().cpu()
vals, idx = logits.topk(K, dim=-1)
print('tokens', logits.shape[0], 'compile_s', round(compile_s, 1))

# topk must be sorted descending by value
assert torch.all(vals[:, :-1] >= vals[:, 1:]), 'topk not sorted'
assert (idx.sort(dim=-1).values.diff(dim=-1) > 0).all(), 'topk has duplicates'

import numpy as np
rec = np.zeros(logits.shape[0], dtype=[('idx', '<i4', (K,)), ('val', '<f2', (K,)), ('off', '<i4')])
rec['idx'] = idx.numpy().astype('<i4')
rec['val'] = vals.numpy().astype('<f2')
rec['off'] = np.array([o[0] for o in offsets], dtype='<i4')
assert (np.diff(rec['off']) >= 0).all(), 'offsets must be non-decreasing'
rec.tofile(out_dir / 'part-00000.bin')

manifest = {
    'teacher': TEACHER, 'k': K, 'tokens': int(logits.shape[0]),
    'vocab': int(len(tok)), 'dtype': 'idx:int32,val:bf16,off:int32',
    'total_bytes': int((out_dir / 'part-00000.bin').stat().st_size),
    'tokenizer': tok.name_or_path,
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
